# Prompt Engineering 101 - Part V. HOMEWORK
## Knowledge & Synthesis (RAG)

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini API with Wrapper)
!pip install -q -U google-genai

from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.0-flash'):
        self.client = genai.Client(api_key=api_key)
        self.models = {
            'gemini-2.0-flash': True,
            'gemini-1.5-flash': True,
            'gemini-1.5-flash-8b': True,
        }
        if preferred_model not in self.models:
            self.models = {preferred_model: True, **self.models}
        self.current_model = preferred_model

    def _get_next_available_model(self):
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
        raise RuntimeError("❌ All available models are currently exhausted.")

    def generate_content(self, contents, config=None):
        try:
            return self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Switching...")
                self.models[self.current_model] = False
                self._get_next_available_model()
                return self.generate_content(contents, config)
            raise e

try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.0-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")

# 🏠 Homework: The Due Diligence Bot

### The Scenario
You are analyzing a startup for potential investment.
You have two sources of information:
1.  **The Pitch Deck (Text):** Claims they are the market leader.
2.  **The News (Search):** Real-world articles about their competitors.

### The Task
Write a Python script using our `model` wrapper that:
1.  **Ingests** a "Fake Pitch Deck" (text string) claiming they have 0 competitors.
2.  **Uses Google Search Tool** to find real competitors for "AI-powered Toaster".
3.  **Synthesizes** a report highlighting the discrepancy between the Pitch (0 competitors) and Reality (Search Results).

### Submission
Submit the Python code and the final "Risk Report".

In [ ]:
# @title (Hidden) Answer Key
pitch_deck = "Our product, the SmartToast AI, is the FIRST and ONLY AI toaster in the world."

investigator_prompt = f"""
CLAIM FROM PITCH DECK: "{pitch_deck}"

TASK:
1. Search Google for "AI powered toaster competitors".
2. Compare the search results to the claim.
3. Is the claim "First and Only" true? Write a Risk Assessment.
"""

# Enable Search
search_tool = Tool(google_search=GoogleSearch())
config = GenerateContentConfig(tools=[search_tool])

print(model.generate_content(investigator_prompt, config=config).text)